# pycalphad + Scheil Demo Notebook

This notebook demonstrates CALPHAD workflows using the repository database:
`Cr-Fe-Ni_miettinen1999.tdb`.

It includes:
1. A Gibbs energy curve
2. A binary Fe-Cr phase diagram
3. A ternary Fe-Cr-Ni phase diagram
4. A single equilibrium calculation and output breakdown
5. A single-composition Scheil solidification calculation


In [ ]:
# If needed, install dependencies in your environment:
# %pip install pycalphad scheil matplotlib numpy xarray


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from pycalphad import Database, calculate, equilibrium, binplot, ternplot
import pycalphad.variables as v
from scheil import simulate_scheil_solidification


In [ ]:
repo_root = Path.cwd()
db_path = repo_root / 'Cr-Fe-Ni_miettinen1999.tdb'

dbf = Database(str(db_path))
components = ['FE', 'CR', 'NI', 'VA']
phases = sorted(dbf.phases.keys())

print(f'Database loaded: {db_path}')
print(f'Number of phases: {len(phases)}')
print('Example phases:', phases[:10])


## 1) Gibbs energy curve (LIQUID phase)

Compute and plot molar Gibbs energy of the liquid phase as a function of temperature
for a fixed alloy composition.


In [ ]:
gm_calc = calculate(
    dbf,
    components,
    'LIQUID',
    output='GM',
    T=np.linspace(1000, 2000, 150),
    P=101325,
    N=1,
    X_CR=0.20,
    X_NI=0.30,
)

T_vals = np.ravel(gm_calc.T.values)
GM_vals = np.ravel(gm_calc.GM.values)

plt.figure(figsize=(7, 4.5))
plt.plot(T_vals, GM_vals)
plt.xlabel('Temperature (K)')
plt.ylabel('Gibbs Energy, GM (J/mol)')
plt.title('LIQUID Gibbs Energy Curve (Fe-20Cr-30Ni)')
plt.grid(True, alpha=0.3)
plt.show()


## 2) Binary Fe-Cr phase diagram

Generate a binary isobaric phase diagram in the Fe-Cr subsystem (Ni fixed to zero).


In [ ]:
binary_conds = {
    v.P: 101325,
    v.T: (800, 2200, 20),
    v.X('CR'): (0, 1, 0.01),
    v.X('NI'): 0,
}

plt.figure(figsize=(7, 5))
binplot(dbf, components, phases, binary_conds)
plt.title('Binary Fe-Cr Phase Diagram (Ni=0)')
plt.show()


## 3) Ternary Fe-Cr-Ni phase diagram

Generate an isothermal ternary section at 1300 K.


In [ ]:
ternary_conds = {
    v.P: 101325,
    v.T: 1300,
    v.X('CR'): (0, 1, 0.01),
    v.X('NI'): (0, 1, 0.01),
}

plt.figure(figsize=(7.5, 6.5))
ternplot(dbf, components, phases, ternary_conds)
plt.title('Ternary Fe-Cr-Ni Phase Diagram at 1300 K')
plt.show()


## 4) Single equilibrium calculation + output breakdown

Run one equilibrium at fixed composition and inspect the returned xarray Dataset.


In [ ]:
eq = equilibrium(
    dbf,
    components,
    phases,
    {
        v.P: 101325,
        v.T: 1200,
        v.N: 1,
        v.X('CR'): 0.20,
        v.X('NI'): 0.30,
    },
)

eq


In [ ]:
print('--- Dataset structure ---')
print('Dimensions:', dict(eq.dims))
print('Coordinates:', list(eq.coords))
print('Data variables:', list(eq.data_vars))

print('
--- Key outputs ---')
print('Total Gibbs energy (J/mol):', float(np.ravel(eq.GM.values)[0]))

phase_names = np.ravel(eq.Phase.values)
phase_fractions = np.ravel(eq.NP.values)

stable = [(str(name), float(frac)) for name, frac in zip(phase_names, phase_fractions) if frac > 1e-8]
print('Stable phases and phase fractions (NP):')
for name, frac in stable:
    print(f'  {name}: {frac:.6f}')


## 5) Single-composition Scheil calculation

Run Scheil solidification for one alloy composition and inspect key outputs.


In [ ]:
composition = {v.X('CR'): 0.20, v.X('NI'): 0.30}

try:
    scheil_result = simulate_scheil_solidification(
        dbf,
        components,
        phases,
        composition,
        start_temperature=1800,
        step_temperature=2,
    )
except TypeError:
    # Fallback for alternate keyword-based API variants
    scheil_result = simulate_scheil_solidification(
        database=dbf,
        components=components,
        phases=phases,
        composition=composition,
        start_temperature=1800,
        step_temperature=2,
    )

print('Scheil result type:', type(scheil_result))
print('Available attributes:', [a for a in dir(scheil_result) if not a.startswith('_')][:20])


In [ ]:
# Typical attributes in scheil results include temperature and liquid-fraction histories.
# The checks below keep the notebook robust across package versions.

temperatures = getattr(scheil_result, 'temperatures', None)
liquid_fraction = getattr(scheil_result, 'fraction_liquid', None)

if temperatures is not None and liquid_fraction is not None:
    plt.figure(figsize=(7, 4.5))
    plt.plot(np.array(temperatures), np.array(liquid_fraction))
    plt.xlabel('Temperature (K)')
    plt.ylabel('Liquid fraction')
    plt.title('Scheil Solidification (Fe-20Cr-30Ni)')
    plt.gca().invert_xaxis()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print('Plot skipped: this scheil version does not expose temperatures/fraction_liquid attributes with these names.')
